In [1]:
!pip install pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("Movies Data:")
display(movies.head())

print("Ratings Data:")
display(ratings.head())

Movies Data:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Data:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
data = pd.merge(ratings, movies, on='movieId')

print("Merged Data:")
display(data.head())

Merged Data:


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [6]:
user_movie_matrix = data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

# Fill missing values with 0
user_movie_matrix = user_movie_matrix.fillna(0)

print("User-Movie Matrix Shape:", user_movie_matrix.shape)

User-Movie Matrix Shape: (610, 9719)


In [7]:
similarity = cosine_similarity(user_movie_matrix)

# Convert to DataFrame for easier use
similarity_df = pd.DataFrame(
    similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

print("Similarity Matrix Ready ✅")

Similarity Matrix Ready ✅


In [8]:
def recommend_movies(user_id, num_recommendations=5):

    if user_id not in user_movie_matrix.index:
        return "User not found"

    # Get similar users
    similar_users = similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Get movies watched by target user
    user_movies = user_movie_matrix.loc[user_id]
    watched_movies = user_movies[user_movies > 0].index.tolist()

    # Collect recommendations
    recommendations = []

    for sim_user in similar_users.index:
        sim_user_movies = user_movie_matrix.loc[sim_user]
        sim_user_movies = sim_user_movies[sim_user_movies > 0]

        for movie in sim_user_movies.index:
            if movie not in watched_movies:
                recommendations.append(movie)

    return list(set(recommendations))[:num_recommendations]

In [9]:
print("Recommended Movies for User 1:")
print(recommend_movies(1))

Recommended Movies for User 1:
['Screamers (1995)', 'Teenage Mutant Ninja Turtles (1990)', 'Bodyguard, The (1992)', 'Batman & Robin (1997)', 'Children of the Corn (1984)']


In [10]:
movie_user_matrix = user_movie_matrix.T

# Compute similarity between movies
movie_similarity = cosine_similarity(movie_user_matrix)

movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

def recommend_similar_movies(movie_name, num_recommendations=5):

    if movie_name not in movie_similarity_df.columns:
        return "Movie not found"

    similar_scores = movie_similarity_df[movie_name].sort_values(ascending=False)[1:num_recommendations+1]

    return similar_scores.index.tolist()


In [11]:
print("\nBecause you watched Toy Story (1995):")
print(recommend_similar_movies("Toy Story (1995)"))



Because you watched Toy Story (1995):
['Toy Story 2 (1999)', 'Jurassic Park (1993)', 'Independence Day (a.k.a. ID4) (1996)', 'Star Wars: Episode IV - A New Hope (1977)', 'Forrest Gump (1994)']


In [12]:
import pickle

pickle.dump(movie_similarity_df, open("movie_similarity.pkl", "wb"))

print("\nModel saved as movie_similarity.pkl")


Model saved as movie_similarity.pkl
